# micrograd examples

In [ ]:
import random
from micrograd import Value, MLP
from micrograd.viz import draw_dot

random.seed(42)

## 1. Backprop by hand

In [ ]:
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a * b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label='f')
L = d * f; L.label = 'L'

L.grad = 1.0
f.grad = d.data
d.grad = f.data
e.grad = d.grad * 1.0
c.grad = d.grad * 1.0
a.grad = e.grad * b.data
b.grad = e.grad * a.data
draw_dot(L)

In [ ]:
print('L before:', L.data)

a.data += a.grad * 0.01
b.data += b.grad * 0.01
c.data += c.grad * 0.01
f.data += f.grad * 0.01
e = a * b
d = e + c
L = d * f
print('L after: ', L.data)

## 2. A single neuron

In [ ]:
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')
x1w1 = x1 * w1; x1w1.label='x1*w1'
x2w2 = x2 * w2; x2w2.label='x2*w2'
x1w1_plus_x2w2 = x1w1 + x2w2; x1w1_plus_x2w2.label='x1*w1 + x2*w2'
n = x1w1_plus_x2w2 + b; n.label='n'
o = n.tanh(); o.label = "o"
draw_dot(o)

In [ ]:
o.backward()
draw_dot(o)

## 3. The same neuron with tanh broken into smaller operations

In [ ]:
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')
x1w1 = x1 * w1; x1w1.label='x1*w1'
x2w2 = x2 * w2; x2w2.label='x2*w2'
x1w1_plus_x2w2 = x1w1 + x2w2; x1w1_plus_x2w2.label='x1*w1 + x2*w2'
n = x1w1_plus_x2w2 + b; n.label='n'
e = (2*n).exp(); e.label = "e"
o = (e-1)/(e+1); o.label = "o"
o.backward()
draw_dot(o)

## 4. Checking against PyTorch

In [ ]:
import torch

x1 = torch.Tensor([2.0]).double(); x1.requires_grad = True
x2 = torch.Tensor([0.0]).double(); x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double(); w1.requires_grad = True
w2 = torch.Tensor([1.0]).double(); w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double(); b.requires_grad = True
n = x1 * w1 + x2 * w2 + b
o = torch.tanh(n)
print(o.data.item())
o.backward()
print("----")
print(x1.grad.item())
print(x2.grad.item())
print(w1.grad.item())
print(w2.grad.item())

## 5. Forward pass through an MLP

In [ ]:
x = [2.0, 3.0]
n = MLP(2, [3, 4, 1])
"""
#When I run something like this what happens is that the first layers are created w 3 and 4 neurons respectively
first layer takes the input with w1,w2 and creates 3 outputs corresponding to 3 o/ps in layer 2
second layer takes the output of this layer of 3 neurons and inputs them into the layer of 4 neurons
last layer combines them in 1 neuron and gives the output
"""
n(x)

In [ ]:
draw_dot(n(x))

## 6. Training an MLP

In [ ]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

n = MLP(3, [4, 4, 1])
print(len(n.parameters()), 'parameters')

In [ ]:
for k in range(100):
    #FORWARD PASS
    ypred = [n(x) for x in xs]
    L = sum((ypi - ysi)**2 for ysi, ypi in zip(ys, ypred))
    #BACKWARD PASS
    for p in n.parameters():
        p.grad = 0.0
    L.backward()
    #PARAM UPDATE
    for p in n.parameters():
        p.data += -0.01 * p.grad
    if k % 10 == 0:
        print(k, L.data)

print('final loss:', L.data)

In [ ]:
[round(n(x).data, 3) for x in xs]